# 01. Baseline: Fast and Low-Cost Model

This notebook supports:
- direct prompt editing,
- configurable generation parameters,
- automatic run logging,
- summary table for prompt/parameter comparison.


## Step 0: Set up environment & authenticate


In [ ]:
from setup_env import setup
from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report
from esmt_workshop.experiment_logging import load_experiment_history, log_experiment_run
from esmt_workshop.student_api import process_batch_addresses, process_single_address
from esmt_workshop.load_utils import get_roots
from esmt_workshop.authenticate import authenticate


setup()
roots = get_roots()
authenticate()

PROJECT_ROOT = roots['PROJECT_ROOT']
SRC_DIR = roots['SRC_DIR']

Local environment detected (project root: C:\Users\JK\Documents\Programming\esmt-workshop)
Setup complete!
PROJECT_ROOT = c:\Users\JK\Documents\Programming\esmt-workshop


## Step 1: Generation Controls


In [ ]:

# Notebook metadata used in experiment history logs.
NOTEBOOK_NAME = '01_baseline_fast_model'

# Students only provide email; proxy endpoint details are managed by organizers.
STUDENT_EMAIL = os.getenv('WORKSHOP_EMAIL', "")

# Keep True for offline demo; set False when organizer-managed proxy is available.
MOCK_MODE = True

# Exposed generation controls available in every processing call.
TEMPERATURE = 0.0
TOP_P = 1.0
TOP_K = 40
MAX_TOKENS = 512
MAX_WORKERS = 8

# Free-text note for this run (optional, appears in history table).
RUN_NOTES = ''

STAGE_NAME = 'baseline'
MODEL_NAME = 'gemini-2.5-flash'
COUNTRY_MODEL = ''
USE_GUARDRAILS = False


## Step 2: Edit Prompt Directly in Notebook


In [ ]:
# Set to False to use built-in stage prompt logic.
USE_CUSTOM_PROMPT = True
PROMPT_LABEL = 'notebook_editable_prompt_v1'
PROMPT_TEMPLATE = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
- Keep values concise and field-specific.
- Prefer exact spans from the input.
- Country Code must be ISO alpha-2 uppercase.
- Use empty strings when unknown.
'''

custom_prompt = PROMPT_TEMPLATE if USE_CUSTOM_PROMPT else None
print('Prompt length:', len(PROMPT_TEMPLATE) if custom_prompt else 0)


## Step 3: Run Pipeline


In [ ]:
import time
import pandas as pd

dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')
input_df = dev_df

# Execute configured stage and measure runtime.
t0 = time.perf_counter()

pred_df = process_batch_addresses(
    input_df,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    country_model=COUNTRY_MODEL if COUNTRY_MODEL else None,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=custom_prompt,
    kb_csv_path=str(PROJECT_ROOT / 'data/input/address_formats.csv'),
    max_workers=MAX_WORKERS,
    mock_mode=MOCK_MODE,
)
runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))
pred_df.head(5)


## Step 4: Evaluate Output


In [ ]:
report = evaluate_predictions(pred_df, input_df, email=STUDENT_EMAIL)
print(report['summary'])
display(report['field_metrics'])


## Step 5: Save Artifacts and Log the Run


In [ ]:
out_dir = PROJECT_ROOT / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

pred_path = out_dir / '01_baseline_fast_model_predictions.csv'
pred_df.to_csv(pred_path, index=False)

report_dir = out_dir / 'report_01_baseline_fast_model'
if report.get('summary'):
    save_evaluation_report(report, report_dir)

run_record = log_experiment_run(
    output_root=PROJECT_ROOT / 'outputs',
    notebook_name=NOTEBOOK_NAME,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    country_model=COUNTRY_MODEL if COUNTRY_MODEL else None,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    max_workers=MAX_WORKERS,
    use_guardrails=USE_GUARDRAILS,
    mock_mode=MOCK_MODE,
    kb_csv_path=str(PROJECT_ROOT / 'data/input/address_formats.csv'),
    prompt_template=custom_prompt,
    prompt_label=PROMPT_LABEL if USE_CUSTOM_PROMPT else 'stage_default_prompt',
    runtime_sec=runtime_sec,
    metrics_summary=report.get('summary', {}),
    notes=RUN_NOTES,
    predictions_path=pred_path,
    report_dir=report_dir if report.get('summary') else '',
)

print('Saved predictions:', pred_path)
if report.get('summary'):
    print('Saved report:', report_dir)
print('Logged run_id:', run_record['run_id'])


## Step 6: Prompt/Parameter History Summary


In [8]:
history_df = load_experiment_history(output_root=PROJECT_ROOT / 'outputs', notebook_name=NOTEBOOK_NAME)
if history_df.empty:
    print('No runs logged yet.')
else:
    cols = [
        'created_at_utc', 'stage', 'model', 'country_model', 'prompt_label',
        'temperature', 'top_p', 'top_k', 'micro_accuracy', 'runtime_sec',
        'row_exact_match', 'prompt_hash', 'notes'
    ]
    display(history_df[cols].head(100))


,created_at_utc,stage,model,country_model,prompt_label,temperature,top_p,top_k,micro_accuracy,runtime_sec,row_exact_match,prompt_hash,notes
0,2026-02-20T18:05:48.075778+00:00,baseline,gemini-2.5-flash,,notebook_editable_prompt_v1,0.0,1.0,40,0.4253,0.0132,0.1,c42d50c83681b77a,


## Assignment

1. Identify common baseline failure patterns.
2. Tune prompt and generation settings.
3. Continue with 02_prompt_tuning.ipynb.
